# Homework 1

In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [2]:
# Pull the lesson pages from the course history

from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    # only check for markdown files
    allowed_extensions={"md"},
    # only grab the markdown files from the lessons
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
len(files)

72

In [4]:
documents = []

for file in files:
    # parse() method returns a dictionary with its filename and content:
    doc = file.parse()
    documents.append(doc)

## Question 1: How many lesson pages?

In [5]:
# Final answer - 72
len(documents)

72

### Q1 Answer: 72 pages

## Question 2: What's the `filename` of the first result?

In [6]:
from ingest_homework import build_index

In [7]:
question = 'How does the agentic loop keep calling the model until it stops?'

In [8]:
index = build_index(documents)

In [9]:
results = index.search(question)

In [10]:
print(results[0]['filename'])

01-agentic-rag/lessons/14-agentic-loop.md


### Q2 answer: 01-agentic-rag/lessons/14-agentic-loop.md

## Question 3: How many input (prompt) tokens did we send to the model for this request?

In [11]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-06-22 04:23:18--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py.1’

rag_helper.py.1     100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-22 04:23:18 (29.2 MB/s) - ‘rag_helper.py.1’ saved [2134/2134]



**Dislaimer: I renamed it to `rag_helper_homework.py` cause I already have `rag_helper.py`**

In [12]:
from rag_helper_homework import RAGBase

In [13]:
question = 'How does the agentic loop keep calling the model until it stops?'

In [14]:
index = build_index(documents)

In [15]:
assistant = RAGBase(index, openai_client)

In [16]:
assistant.rag(question)

('It keeps calling the model inside a `while True` loop, and after each response it checks whether the model returned any `function_call` items.\n\n- If there are function calls, the code runs the tool, appends the tool output to the message history, and loops again.\n- If there are no function calls, it breaks out of the loop and stops.\n\nSo the stopping condition is simply: **no function calls in the model’s response**.',
 ResponseUsage(input_tokens=7121, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=96, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7217))

In [17]:
answer, usage = assistant.rag(question)
print(usage.input_tokens)

7121


### Q3 Answer: about 7000 tokens

## Question 4: How many chunks do you get?

In [18]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [19]:
len(chunks)

295

### Q4 Answer: 295 chunks

## Q5: Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?

In [20]:
chunk_index = build_index(chunks)
chunk_assistant = RAGBase(chunk_index, openai_client)

In [21]:
answer, usage = chunk_assistant.rag(question)
print(usage.input_tokens)

2304


### Q5 Answer: This time we have about 2300 tokens as opposed to initial 7000 tokens, so about 3 times less

## Q6. How many times did the agent call search

In [22]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [23]:
instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
""".strip()

In [24]:
question = "How does the agentic loop work, and how is it different from plain RAG?"

In [ ]:
# add a type hint and a docstring to search for ToyAIKit to 
# read and derive the schema
# COME BACK
def search(query: str) -> dict[str, str]:
    """
    Search the course lesson chunks for information relevant to the query.
    """
    boost_dict = {"content": 3.0}
    # chunk_index
    return chunk_index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [26]:
# register the tool without passing the search_tool description to it
agent_tools = Tools()
agent_tools.add_tool(search)

In [27]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the course lesson chunks for information relevant to the query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [28]:

# define the actual agent
# chat_interface displays the output
chat_interface = IPythonChatInterface()
# every time somethign happens, like a tool call or the message is displayed, 
# the callback is going to be invoked to display things 
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    # chat interface is used for debugging purposes
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [29]:
# uses iPython Jupyter notebook widgets
result = runner.loop(
    prompt=question,
    callback=callback,
)

-> Response received


-> Response received


### Q6: I got 3 to 4 calls to search agent